# Data Engineering — Moduł 1: Wprowadzenie do inżynierii danych

To piąty kurs w tej serii — po Pythonie, SQL/PostgreSQL, Statystyce i Databricks/PySpark.
W poprzednich czterech kursach nauczyłeś/aś się PRACOWAĆ Z danymi: czyścić je, analizować,
modelować statystycznie i przetwarzać na dużą skalę w Sparku. Ten kurs jest inny — uczy,
jak zbudować i UTRZYMYWAĆ cały **potok danych** (data pipeline), który dostarcza dane od
źródła do analityka/modelu ML niezawodnie, cyklicznie i bez ręcznej interwencji. To
właśnie robi na co dzień **inżynier danych** (Data Engineer).

## Spis treści
1. [Czym jest inżynieria danych?](#sec1)
2. [ETL kontra ELT](#sec2)
3. [Przetwarzanie wsadowe kontra strumieniowe](#sec3)
4. [Anatomia nowoczesnego stosu danych](#sec4)
5. [Role w zespole danych: Analyst, Engineer, Scientist](#sec5)
6. [Środowisko tego kursu](#sec6)
7. [Podsumowanie i ćwiczenia](#sec7)


<a id="sec1"></a>
## 1. Czym jest inżynieria danych?

**Inżynieria danych** to dyscyplina budowania systemów, które przenoszą i przekształcają
dane -- od surowych źródeł (bazy transakcyjne, API, pliki, zdarzenia z aplikacji) aż po
formę gotową do analizy, raportowania czy trenowania modeli ML. Kluczowe pytania, na
które odpowiada inżynier danych, to nie "co te dane mówią" (to pytanie analityka /
data scientist), tylko:

- Jak REGULARNIE i NIEZAWODNIE dostarczyć świeże dane (codziennie? co godzinę? w czasie
  rzeczywistym?).
- Jak wykryć i obsłużyć błędy, zanim zepsują raport albo model (walidacja, alerty).
- Jak zaprojektować dane tak, żeby dziesiątki osób w firmie mogły z nich korzystać, nie
  psując się nawzajem (modelowanie, kontrakty danych, uprawnienia).
- Jak to wszystko przeskaluje się, gdy danych będzie 100x więcej.

Wszystko, czego się nauczyliśmy do tej pory -- SQL (transformacje, okna, JOIN-y), Python
(automatyzacja, `pandas`), Spark/Databricks (skala, MLlib) -- to NARZĘDZIA, których
inżynier danych używa. Ten kurs dodaje brakujący element: jak te narzędzia SPIĄĆ w
działający, zautomatyzowany system.

> ⚡ **Inżynieria danych to "DevOps dla danych"**
>
> Jeśli znasz pojęcie DevOps (automatyzacja wdrażania i utrzymania oprogramowania) -- inżynieria danych to bardzo podobna filozofia, zastosowana do POTOKÓW DANYCH zamiast do aplikacji: automatyzacja, wersjonowanie, testowanie, monitoring, odzyskiwanie po awarii.

<a id="sec2"></a>
## 2. ETL kontra ELT

Klasyczny akronim **ETL** (*Extract, Transform, Load*) opisuje kolejność: wyciągamy dane
ze źródła, PRZEKSZTAŁCAMY je (czyszczenie, agregacje, łączenie) na osobnym serwerze
przetwarzania, a dopiero POTEM ładujemy gotowy wynik do hurtowni danych. To podejście
dominowało, gdy hurtownie danych miały ograniczoną moc obliczeniową, więc transformacje
trzeba było robić "na boku".

Nowoczesne hurtownie/lakehouse'y (Snowflake, BigQuery, Databricks -- poznany w
poprzednim kursie) mają OGROMNĄ moc obliczeniową wbudowaną. Stąd popularność **ELT**
(*Extract, Load, Transform*): najpierw ładujemy surowe dane WPROST do hurtowni (szybko,
bez przekształceń), a transformacje robimy PÓŹNIEJ, wewnątrz hurtowni, zwykłym SQL-em --
dokładnie to, do czego służy **dbt** (Moduły 4-5 tego kursu).

| | ETL | ELT |
|---|---|---|
| Transformacja | Przed załadowaniem, na osobnym serwerze | Po załadowaniu, wewnątrz hurtowni |
| Surowe dane | Zwykle NIE są zachowywane | Zachowane w warstwie bronze (Moduł 11 kursu Databricks) |
| Narzędzia | Informatica, SSIS, custom Python | dbt, Spark SQL, SQL hurtowni |
| Kiedy dominuje | Starsze/on-premise systemy | Nowoczesny stos w chmurze |

W tym kursie idziemy w stronę ELT -- dokładnie tak, jak robiliśmy to nieświadomie już w
Module 15 poprzedniego kursu (bronze → silver → gold, transformacje SQL-em WEWNĄTRZ
Sparka, a nie na zewnętrznym serwerze ETL).

<a id="sec3"></a>
## 3. Przetwarzanie wsadowe kontra strumieniowe

- **Batch (wsadowe)** -- przetwarzamy dane W PACZKACH, w ustalonych odstępach czasu (np.
  "raz dziennie o 2:00 w nocy przelicz wczorajsze zamówienia"). Prostsze w budowie i
  debugowaniu, wystarczające dla większości raportów biznesowych. Cały kurs Databricks
  (poza jedną sekcją) i większość tego kursu (Airflow, dbt) dotyczy przetwarzania
  wsadowego.
- **Streaming (strumieniowe)** -- przetwarzamy KAŻDE zdarzenie osobno, w miarę jak
  napływa (milisekundy/sekundy po jego wystąpieniu). Potrzebne tam, gdzie liczy się
  natychmiastowa reakcja: wykrywanie oszustw przy płatności, personalizacja w czasie
  rzeczywistym, monitoring systemów. Poznamy je w Modułach 8-9 (Kafka, Spark Structured
  Streaming).

Coraz popularniejsze jest podejście **hybrydowe** (czasem zwane *Lambda* albo
*Kappa architecture*): większość danych płynie przez batch (tanio, prosto), a tylko
te przypadki, które NAPRAWDĘ potrzebują natychmiastowej reakcji, idą przez streaming.

> ⚠️ **Streaming nie jest "lepszy" -- jest DROŻSZY i TRUDNIEJSZY**
>
> Częsty błąd początkujących: zakładać, że streaming to zawsze lepszy wybór, bo "szybszy". W praktyce systemy strumieniowe są znacznie trudniejsze w budowie, debugowaniu i utrzymaniu niż batch (błąd w logice batch naprawiasz i przeliczasz ponownie; błąd w logice streamingu mógł już wysłać złe dane dalej). Wybieraj streaming tylko wtedy, gdy opóźnienie rzędu godzin/minut faktycznie szkodzi biznesowi.

<a id="sec4"></a>
## 4. Anatomia nowoczesnego stosu danych

Typowy, nowoczesny stos danych (*modern data stack*) składa się z warstw, które poznamy
po kolei w tym kursie:

1. **Źródła danych** -- bazy transakcyjne aplikacji (jak PostgreSQL z kursu SQL), API,
   pliki, zdarzenia z aplikacji (kliknięcia, logi).
2. **Ingestia (wczytywanie)** -- narzędzia kopiujące dane ze źródeł do magazynu
   (Fivetran/Airbyte w praktyce; w tym kursie robimy to sami, tak jak w Module 15
   poprzedniego kursu).
3. **Przechowywanie** -- data lake / warehouse / lakehouse (poznane dogłębnie w Module 11
   kursu Databricks) -- tu ląduje zarówno surowa, jak i przekształcona dane.
4. **Transformacje** -- SQL/dbt (Moduły 4-5, 7 tego kursu) przekształcający surowe dane
   (bronze) w czyste, zamodelowane tabele (silver/gold) -- ELT z sekcji 2.
5. **Orkiestracja** -- Airflow (Moduły 2-3 tego kursu) decyduje, KIEDY i W JAKIEJ
   KOLEJNOŚCI uruchamiają się powyższe kroki, obsługuje błędy i retry.
6. **Streaming** (opcjonalnie, dla przypadków czasu rzeczywistego) -- Kafka + Spark
   Structured Streaming (Moduły 8-9).
7. **Konsumpcja** -- narzędzia BI (Tableau/Power BI), notebooki analityczne, modele ML
   (MLlib z poprzedniego kursu) -- korzystają z gotowych danych z warstwy gold.

Moduł 10 (projekt końcowy) połączy warstwy 4, 5 i 6 w jeden działający potok.

> 🏭 **To NIE jest jeden produkt**
>
> "Nowoczesny stos danych" to zestaw WSPÓŁPRACUJĄCYCH, ale OSOBNYCH narzędzi -- każda warstwa to inna firma/projekt open source (Airflow do orkiestracji, dbt do transformacji, Snowflake/Databricks do przechowywania, Kafka do streamingu). Umiejętność SKŁADANIA tych klocków w spójny system to sedno pracy inżyniera danych.

<a id="sec5"></a>
## 5. Role w zespole danych: Analyst, Engineer, Scientist

Te trzy role często się mieszają w mniejszych firmach, ale warto rozumieć różnicę:

- **Data Analyst** -- odpowiada na pytania biznesowe za pomocą GOTOWYCH danych (SQL,
  Excel, BI). To, czego uczył kurs SQL i częściowo Statystyka.
- **Data Scientist** -- buduje modele statystyczne/ML, żeby PRZEWIDYWAĆ albo
  wyjaśniać zjawiska. To, czego uczył kurs Statystyki i sekcje MLlib kursu Databricks.
- **Data Engineer** -- buduje i utrzymuje INFRASTRUKTURĘ, która dostarcza dane obu
  powyższym rolom: niezawodnie, na czas, w odpowiednim kształcie. To jest ten kurs.

W praktyce granice się zacierają -- dobry Data Analyst/Scientist zna podstawy inżynierii
danych (i odwrotnie), a w mniejszych zespołach jedna osoba pełni wszystkie trzy role. Ten
kurs domyka Twoje kompetencje właśnie w tym kierunku.

<a id="sec6"></a>
## 6. Środowisko tego kursu

Cztery główne narzędzia tego kursu i jak z nich korzystamy lokalnie:

- **Apache Airflow** -- zainstalowany w izolowanym środowisku Python (venv) z lokalną
  bazą metadanych SQLite. Będziemy uruchamiać DAG-i komendą `airflow dags test` --
  dokładnie tę samą składnię DAG-ów, której użyłbyś na produkcyjnym klastrze Airflow.
- **dbt (data build tool)** -- z adapterem **DuckDB** (lekka, wbudowana baza
  analityczna, plikowa jak SQLite, ale zoptymalizowana pod zapytania analityczne SQL).
  W prawdziwym projekcie zamieniłbyś DuckDB na Snowflake/BigQuery/Databricks -- kod dbt
  (modele SQL) zostaje niemal identyczny.
- **Apache Kafka** -- w tym środowisku sandboksowym NIE mamy dostępu do uruchomienia
  prawdziwego brokera Kafki (brak Dockera z uruchomionym demonem, brak dostępu
  sieciowego do pobrania binarek Kafki) -- dokładnie ta sama sytuacja co z Delta Lake w
  poprzednim kursie. Moduł 8 opisze Kafkę koncepcyjnie, z przykładami kodu do
  wypróbowania na własnym środowisku (lub Confluent Cloud / Databricks).
- **Spark Structured Streaming** -- URUCHAMIALNE lokalnie (ten sam Spark z poprzedniego
  kursu), używając wbudowanego źródła `rate` jako bezpiecznego zamiennika prawdziwego
  strumienia Kafki na potrzeby nauki (Moduł 9).

> ⚠️ **Dlaczego nie "prawdziwa" Kafka?**
>
> To środowisko chmurowe (sandbox) nie ma uruchomionego demona Dockera ani dostępu sieciowego do pobrania binarek spoza kilku dozwolonych rejestrów pakietów (pip, npm) -- nie da się więc postawić lokalnego brokera Kafki. To ograniczenie środowiska, NIE luka w samej Kafce jako technologii -- traktujemy Moduł 8 analogicznie do Modułu 10 (Delta Lake) z poprzedniego kursu: koncepcyjnie poprawny, ale bez lokalnego wykonania.

<a id="sec7"></a>
## 7. Podsumowanie i ćwiczenia

W tym module poznaliśmy:
- czym jest inżynieria danych i czym różni się od pracy analityka/data scientist,
- różnicę między ETL a ELT (i dlaczego nowoczesny stos preferuje ELT),
- różnicę między przetwarzaniem wsadowym a strumieniowym,
- anatomię nowoczesnego stosu danych: źródła → ingestia → przechowywanie →
  transformacje → orkiestracja → (opcjonalnie) streaming → konsumpcja,
- jakich narzędzi użyjemy w tym kursie i jak są skonfigurowane lokalnie.

### 📝 Ćwiczenie 1: ETL czy ELT?

Dla każdego z poniższych scenariuszy napisz w komentarzu, czy bliżej mu do ETL czy ELT, i uzasadnij jednym zdaniem: (a) mała firma z własnym serwerem SQL Server sprzed 10 lat, (b) firma używająca Snowflake i dbt, (c) potok, który agreguje dane PRZED zapisaniem, bo docelowa baza ma bardzo ograniczoną moc obliczeniową.

<details>
<summary>Pokaż rozwiązanie</summary>

```text
# (a) ETL -- stary serwer SQL Server ma ograniczona moc, transformacje robi sie
#     na osobnym serwerze PRZED zaladowaniem, zeby nie obciazac bazy produkcyjnej.
# (b) ELT -- Snowflake ma ogromna moc obliczeniowa, dbt transformuje dane JUZ
#     zaladowane do Snowflake, czystym SQL-em.
# (c) ETL -- transformacja (agregacja) dzieje sie PRZED zaladowaniem, bo docelowa
#     baza nie udzwignie transformacji na surowych danych.
print("Zobacz komentarze powyzej.")
```

</details>

### 📝 Ćwiczenie 2: Batch czy streaming?

Dla każdego przypadku użycia zdecyduj: batch czy streaming, i uzasadnij: (a) miesięczny raport sprzedaży dla zarządu, (b) wykrywanie prób oszustwa podczas płatności kartą, (c) rekomendacje produktów wyświetlane na stronie sklepu, (d) obliczanie premii kwartalnych dla sprzedawców.

<details>
<summary>Pokaż rozwiązanie</summary>

```text
# (a) batch -- raport miesieczny nie wymaga danych sprzed sekundy, wystarczy
#     przeliczyc raz dziennie/tygodniowo.
# (b) streaming -- oszustwo trzeba wykryc I ZABLOKOWAC w czasie transakcji,
#     opoznienie rzedu minut jest bezuzyteczne.
# (c) czesto streaming (albo "blisko-streaming") -- rekomendacje oparte na
#     BIEZACYM zachowaniu uzytkownika dzialaja lepiej w czasie rzeczywistym,
#     ale wiele firm zaczyna od prostszego batchu i dopiero pozniej dokleja streaming.
# (d) batch -- premie kwartalne z definicji licza sie raz na kwartal, streaming
#     nie dalby zadnej korzysci.
print("Zobacz komentarze powyzej.")
```

</details>

### 📝 Ćwiczenie 3: Narysuj (opisz) swój własny stos danych

Wyobraź sobie prosty sklep internetowy (jak nasz przykładowy zestaw danych z poprzednich kursów). Opisz jednym zdaniem na warstwę, co konkretnie znalazłoby się w każdej z sześciu warstw stosu z sekcji 4 (źródła, ingestia, przechowywanie, transformacje, orkiestracja, konsumpcja) dla TEGO konkretnego sklepu.

<details>
<summary>Pokaż rozwiązanie</summary>

```text
# Zrodla: baza PostgreSQL aplikacji sklepu (klienci, zamowienia, produkty).
# Ingestia: codzienny job kopiujacy nowe/zmienione wiersze z PostgreSQL do lakehouse'u.
# Przechowywanie: warstwa bronze (surowe kopie tabel) w Databricks/S3.
# Transformacje: dbt buduje warstwe silver (czyszczenie) i gold (agregaty sprzedazy,
#     wymiary klientow) z warstwy bronze.
# Orkiestracja: DAG Airflow uruchamiany codziennie o 3:00, kolejno: ingestia -> dbt run
#     -> powiadomienie zespolu analityki, ze dane sa gotowe.
# Konsumpcja: dashboard Power BI dla zarzadu + notebook Databricks z modelem MLlib
#     przewidujacym anulowania zamowien (Modul 12-13 poprzedniego kursu).
print("Zobacz komentarze powyzej.")
```

</details>

> 🔥 **Wyzwanie: gdzie streaming faktycznie się opłaca?**
>
> Zaproponuj TRZY konkretne przypadki użycia streamingu (inne niż podane w sekcji 3 i ćwiczeniu 2) w dowolnej branży, którą znasz -- dla każdego uzasadnij jednym zdaniem, DLACZEGO opóźnienie rzędu minut/godzin (czyli batch) faktycznie zaszkodziłoby biznesowi. To ćwiczenie z myślenia, nie z kodu -- zapisz odpowiedzi jako komentarze.

<details>
<summary>Pokaż rozwiązanie</summary>

```text
# Przykladowe odpowiedzi (Twoje moga byc inne -- liczy sie uzasadnienie):
# 1. Monitoring floty pojazdow (logistyka): opozniona informacja o awarii silnika
#    moze oznaczac, ze kierowca dowie sie o problemie dopiero po utknieciu w trasie.
# 2. Handel algorytmiczny na gieldzie: cena zmienia sie w milisekundach, decyzja
#    podjeta na podstawie danych sprzed minuty jest juz nieaktualna.
# 3. Systemy bezpieczenstwa (wykrywanie wlaman do sieci firmowej): kazda minuta
#    zwloki to wiecej czasu dla atakujacego na eskalacje dostepu.
print("Zobacz komentarze powyzej.")
```

Klucz do dobrej odpowiedzi: streaming ma sens tylko wtedy, gdy KTOS (czlowiek albo system) faktycznie MOZE i BEDZIE dzialac na podstawie danych w ciagu sekund/minut -- jesli i tak nikt nie zareaguje szybciej niz raz dziennie, streaming to niepotrzebny koszt.

</details>

## Co dalej?

W **Module 2** zaczynamy praktykę: instalujemy i uruchamiamy Apache Airflow lokalnie,
piszemy pierwszy DAG i poznajemy podstawowe klocki orkiestracji (Task, Operator,
harmonogram, zależności).